In [122]:
# ============================================================
# POR-DASHBOARD
# FULL SYSTEM STRESS TEST
# JUPYTER-COMPATIBLE VERSION
# ============================================================

import sys
import os
import importlib
import numpy as np
import pandas as pd


# ============================================================
# 1. PROJECT PATH
# ============================================================

print("=" * 75)
print("POR-DASHBOARD FULL STRESS TEST")
print("=" * 75)

PROJECT_ROOT = os.getcwd()

if os.path.basename(PROJECT_ROOT).lower() == "notebooks":
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("\nCurrent working directory:")
print(os.getcwd())

print("Project root:")
print(PROJECT_ROOT)


# ============================================================
# 2. TEST COUNTERS
# ============================================================

TESTS_RUN = 0
TESTS_PASSED = 0
TESTS_FAILED = 0

FAILURES = []


def test_pass(name):

    global TESTS_RUN
    global TESTS_PASSED

    TESTS_RUN += 1
    TESTS_PASSED += 1

    print(f"    🟢 PASS: {name}")


def test_fail(name, error):

    global TESTS_RUN
    global TESTS_FAILED

    TESTS_RUN += 1
    TESTS_FAILED += 1

    FAILURES.append(
        {
            "test": name,
            "error": f"{type(error).__name__}: {error}"
        }
    )

    print(f"    🔴 FAIL: {name}")
    print(
        f"       {type(error).__name__}: {error}"
    )


def run_test(name, function):

    try:

        function()

        test_pass(name)

        return True

    except Exception as error:

        test_fail(
            name,
            error
        )

        return False


def section(title):

    print()
    print("=" * 75)
    print(title)
    print("=" * 75)


# ============================================================
# 3. IMPORT PROJECT MODULES
# ============================================================

section("1. MODULE IMPORT TEST")

MODULE_NAMES = [
    "src.returns.returns",
    "src.risk.risk",
    "src.risk.portfolio_risk",
    "src.optimization.optimization",
    "src.backtest.backtest",
    "src.performance.performance"
]

MODULES = {}

for module_name in MODULE_NAMES:

    def import_test(
        module_name=module_name
    ):

        module = importlib.import_module(
            module_name
        )

        module = importlib.reload(
            module
        )

        MODULES[
            module_name
        ] = module

        assert module is not None

    run_test(
        module_name,
        import_test
    )


returns_module = MODULES.get(
    "src.returns.returns"
)

risk_module = MODULES.get(
    "src.risk.risk"
)

portfolio_risk_module = MODULES.get(
    "src.risk.portfolio_risk"
)

optimization_module = MODULES.get(
    "src.optimization.optimization"
)

backtest_module = MODULES.get(
    "src.backtest.backtest"
)

performance_module = MODULES.get(
    "src.performance.performance"
)


# ============================================================
# 4. SYNTHETIC LONG-FORMAT DATA
# ============================================================
#
# IMPORTANT
#
# Your Returns and Risk modules expect data containing:
#
# Date
# Ticker
# Close
#
# Therefore we generate LONG FORMAT here.
#
# Example:
#
# Date        Ticker      Close
# 2020-01-01  ASSET_1     100
# 2020-01-02  ASSET_1     101
# 2020-01-03  ASSET_1     102
# ...
#
# ============================================================

def make_long_price_data(
    n_assets=5,
    n_days=1000,
    seed=42
):

    rng = np.random.default_rng(
        seed
    )

    dates = pd.bdate_range(
        start="2020-01-01",
        periods=n_days
    )

    tickers = [
        f"ASSET_{i + 1}"
        for i in range(n_assets)
    ]

    daily_returns = rng.normal(
        loc=0.0003,
        scale=0.012,
        size=(
            n_days,
            n_assets
        )
    )

    prices = (
        100
        * np.exp(
            np.cumsum(
                daily_returns,
                axis=0
            )
        )
    )

    rows = []

    for j, ticker in enumerate(
        tickers
    ):

        for i, date in enumerate(
            dates
        ):

            rows.append(
                {
                    "Date": date,
                    "Ticker": ticker,
                    "Close": prices[i, j]
                }
            )

    return pd.DataFrame(rows)


# ============================================================
# 5. SYNTHETIC WIDE-FORMAT PRICE DATA
# ============================================================
#
# Used for:
#
# Optimization
# Backtesting
# Performance
#
# ============================================================

def make_price_data(
    n_assets=5,
    n_days=1000,
    seed=42
):

    rng = np.random.default_rng(
        seed
    )

    dates = pd.bdate_range(
        start="2020-01-01",
        periods=n_days
    )

    daily_returns = rng.normal(
        loc=0.0003,
        scale=0.012,
        size=(
            n_days,
            n_assets
        )
    )

    prices = (
        100
        * np.exp(
            np.cumsum(
                daily_returns,
                axis=0
            )
        )
    )

    columns = [
        f"ASSET_{i + 1}"
        for i in range(n_assets)
    ]

    return pd.DataFrame(
        prices,
        index=dates,
        columns=columns
    )


# ============================================================
# 6. RETURNS ENGINE
# ============================================================

section("2. RETURNS ENGINE")

if returns_module is not None:

    def test_simple_returns():

        data = make_long_price_data()

        result = (
            returns_module
            .calculate_simple_returns(
                data
            )
        )

        assert isinstance(
            result,
            pd.DataFrame
        )

        assert "Return" in result.columns

        assert "Ticker" in result.columns

        assert "Date" in result.columns

        assert result["Return"].notna().sum() > 0

        valid_returns = (
            result["Return"]
            .dropna()
            .to_numpy(
                dtype=float
            )
        )

        assert np.isfinite(
            valid_returns
        ).all()

    run_test(
        "Simple returns",
        test_simple_returns
    )


    def test_log_returns():

        data = make_long_price_data()

        result = (
            returns_module
            .calculate_log_returns(
                data
            )
        )

        assert isinstance(
            result,
            pd.DataFrame
        )

        assert "Log_Return" in result.columns

        assert "Ticker" in result.columns

        assert "Date" in result.columns

        valid_returns = (
            result["Log_Return"]
            .dropna()
            .to_numpy(
                dtype=float
            )
        )

        assert np.isfinite(
            valid_returns
        ).all()

    run_test(
        "Log returns",
        test_log_returns
    )


    def test_return_matrix():

        data = make_long_price_data()

        result = (
            returns_module
            .create_return_matrix(
                data
            )
        )

        assert isinstance(
            result,
            pd.DataFrame
        )

        assert result.shape[1] == 5

        assert result.index.is_monotonic_increasing

        assert not result.isna().any().any()

        assert np.isfinite(
            result.to_numpy(
                dtype=float
            )
        ).all()

    run_test(
        "Return matrix",
        test_return_matrix
    )


    def test_log_return_matrix():

        data = make_long_price_data()

        result = (
            returns_module
            .create_log_return_matrix(
                data
            )
        )

        assert isinstance(
            result,
            pd.DataFrame
        )

        assert result.shape[1] == 5

        assert result.index.is_monotonic_increasing

        assert not result.isna().any().any()

        assert np.isfinite(
            result.to_numpy(
                dtype=float
            )
        ).all()

    run_test(
        "Log return matrix",
        test_log_return_matrix
    )


    def test_cumulative_returns():

        data = pd.DataFrame(
            {
                "Ticker": [
                    "A",
                    "A",
                    "A",
                    "B",
                    "B",
                    "B"
                ],
                "Return": [
                    0.10,
                    -0.05,
                    0.10,
                    0.05,
                    0.05,
                    0.05
                ]
            }
        )

        result = (
            returns_module
            .calculate_cumulative_returns(
                data
            )
        )

        expected_a = (
            1.10
            * 0.95
            * 1.10
        ) - 1

        actual_a = (
            result.loc[
                result["Ticker"] == "A",
                "Cumulative_Return"
            ].iloc[-1]
        )

        assert np.isclose(
            actual_a,
            expected_a
        )

    run_test(
        "Cumulative return identity",
        test_cumulative_returns
    )


    def test_historical_expected_return():

        returns = pd.DataFrame(
            {
                "ASSET_1": [
                    0.01,
                    0.02,
                    -0.01
                ],
                "ASSET_2": [
                    0.02,
                    0.01,
                    0.03
                ]
            }
        )

        result = (
            returns_module
            .calculate_historical_expected_return(
                returns
            )
        )

        assert isinstance(
            result,
            pd.Series
        )

        assert len(result) == 2

        assert np.isfinite(
            result.to_numpy(
                dtype=float
            )
        ).all()

    run_test(
        "Historical expected return",
        test_historical_expected_return
    )


    def test_geometric_expected_return():

        returns = pd.DataFrame(
            {
                "ASSET_1": [
                    0.01,
                    0.02,
                    -0.01
                ],
                "ASSET_2": [
                    0.02,
                    0.01,
                    0.03
                ]
            }
        )

        result = (
            returns_module
            .calculate_geometric_expected_return(
                returns
            )
        )

        assert isinstance(
            result,
            pd.Series
        )

        assert len(result) == 2

        assert np.isfinite(
            result.to_numpy(
                dtype=float
            )
        ).all()

    run_test(
        "Geometric expected return",
        test_geometric_expected_return
    )


# ============================================================
# 7. RISK ENGINE
# ============================================================

section("3. RISK ENGINE")

if (
    risk_module is not None
    and returns_module is not None
):

    long_data = make_long_price_data(
        n_assets=5,
        n_days=1000,
        seed=100
    )

    long_returns = (
        returns_module
        .calculate_simple_returns(
            long_data
        )
    )

    return_matrix = (
        returns_module
        .create_return_matrix(
            long_data
        )
    )


    def test_historical_volatility():

        result = (
            risk_module
            .calculate_historical_volatility(
                long_returns
            )
        )

        assert isinstance(
            result,
            pd.Series
        )

        assert len(result) == 5

        assert (
            result >= 0
        ).all()

        assert np.isfinite(
            result.to_numpy(
                dtype=float
            )
        ).all()

    run_test(
        "Historical volatility",
        test_historical_volatility
    )


    def test_ewma_volatility():

        result = (
            risk_module
            .calculate_ewma_volatility(
                long_returns
            )
        )

        assert isinstance(
            result,
            pd.Series
        )

        assert len(result) == 5

        assert (
            result >= 0
        ).all()

        assert np.isfinite(
            result.to_numpy(
                dtype=float
            )
        ).all()

    run_test(
        "EWMA volatility",
        test_ewma_volatility
    )


    def test_sample_covariance():

        result = (
            risk_module
            .calculate_sample_covariance(
                long_returns
            )
        )

        assert result.shape == (
            5,
            5
        )

        assert np.allclose(
            result.values,
            result.values.T
        )

        assert np.isfinite(
            result.values
        ).all()

    run_test(
        "Sample covariance",
        test_sample_covariance
    )


    def test_ledoit_wolf_covariance():

        result = (
            risk_module
            .calculate_ledoit_wolf_covariance(
                long_returns
            )
        )

        assert result.shape == (
            5,
            5
        )

        assert np.allclose(
            result.values,
            result.values.T
        )

        assert np.isfinite(
            result.values
        ).all()

    run_test(
        "Ledoit-Wolf covariance",
        test_ledoit_wolf_covariance
    )


    def test_correlation():

        result = (
            risk_module
            .calculate_correlation(
                long_returns
            )
        )

        assert result.shape == (
            5,
            5
        )

        assert np.allclose(
            np.diag(result),
            1.0,
            atol=1e-10
        )

        assert (
            result.values >= -1.000001
        ).all()

        assert (
            result.values <= 1.000001
        ).all()

    run_test(
        "Correlation matrix",
        test_correlation
    )


# ============================================================
# 8. PORTFOLIO RISK ENGINE
# ============================================================

section("4. PORTFOLIO RISK ENGINE")

if portfolio_risk_module is not None:

    returns = make_price_data(
        n_assets=5,
        n_days=1000,
        seed=200
    ).pct_change().dropna()

    covariance = (
        returns.cov()
        * 252
    )

    weights = pd.Series(
        np.ones(5) / 5,
        index=returns.columns,
        dtype=float
    )


    def test_portfolio_variance():

        result = (
            portfolio_risk_module
            .portfolio_variance(
                weights,
                covariance
            )
        )

        manual = (
            weights.to_numpy()
            @ covariance.to_numpy()
            @ weights.to_numpy()
        )

        assert np.isclose(
            result,
            manual,
            atol=1e-10
        )

    run_test(
        "Portfolio variance identity",
        test_portfolio_variance
    )


    def test_portfolio_volatility():

        variance = (
            portfolio_risk_module
            .portfolio_variance(
                weights,
                covariance
            )
        )

        volatility = (
            portfolio_risk_module
            .portfolio_volatility(
                weights,
                covariance
            )
        )

        assert np.isclose(
            volatility,
            np.sqrt(variance),
            atol=1e-10
        )

    run_test(
        "Portfolio volatility identity",
        test_portfolio_volatility
    )


    def test_marginal_risk():

        marginal = (
            portfolio_risk_module
            .marginal_risk(
                weights,
                covariance
            )
        )

        assert len(marginal) == 5

        assert np.isfinite(
            marginal.to_numpy(
                dtype=float
            )
        ).all()

    run_test(
        "Marginal risk",
        test_marginal_risk
    )


    def test_component_risk():

        component = (
            portfolio_risk_module
            .component_risk(
                weights,
                covariance
            )
        )

        portfolio_volatility = (
            portfolio_risk_module
            .portfolio_volatility(
                weights,
                covariance
            )
        )

        assert np.isclose(
            component.sum(),
            portfolio_volatility,
            atol=1e-8
        )

    run_test(
        "Component risk sums to portfolio volatility",
        test_component_risk
    )


    def test_risk_contribution():

        contribution = (
            portfolio_risk_module
            .risk_contribution(
                weights,
                covariance
            )
        )

        assert np.isclose(
            contribution.sum(),
            1.0,
            atol=1e-8
        )

    run_test(
        "Risk contribution sums to 1",
        test_risk_contribution
    )


    def test_weight_concentration():

        result = (
            portfolio_risk_module
            .weight_concentration(
                weights
            )
        )

        assert np.isfinite(
            float(result)
        )

        assert result >= 0

        assert result <= 1

    run_test(
        "Weight concentration",
        test_weight_concentration
    )


    def test_diversification_ratio():

        result = (
            portfolio_risk_module
            .diversification_ratio(
                weights,
                covariance
            )
        )

        assert np.isfinite(
            float(result)
        )

        assert result >= 1.0 - 1e-8

    run_test(
        "Diversification ratio",
        test_diversification_ratio
    )


# ============================================================
# 9. OPTIMIZATION ENGINE
# ============================================================

section("5. OPTIMIZATION ENGINE")

if optimization_module is not None:

    returns, covariance = (
        (
            make_price_data(
                n_assets=5,
                n_days=1000,
                seed=300
            )
            .pct_change()
            .dropna()
        ),
        None
    )

    covariance = (
        returns.cov()
        * 252
    )

    expected_returns = (
        returns.mean()
        * 252
    )

    market_weights = pd.Series(
        np.ones(5) / 5,
        index=returns.columns
    )


    optimizer_functions = {

        "Equal Weight":
            optimization_module.equal_weight,

        "Minimum Variance":
            optimization_module.minimum_variance,

        "Maximum Sharpe":
            optimization_module.maximum_sharpe,

        "Risk Parity":
            optimization_module.risk_parity,

        "HRP":
            optimization_module.hierarchical_risk_parity,

        "Black-Litterman":
            optimization_module.black_litterman
    }


    optimizer_outputs = {}


    for method_name, optimizer in (
        optimizer_functions.items()
    ):

        def optimizer_test(
            method_name=method_name,
            optimizer=optimizer
        ):

            kwargs = {
                "expected_returns":
                    expected_returns,

                "covariance":
                    covariance
            }

            if method_name == "Black-Litterman":

                kwargs[
                    "market_weights"
                ] = market_weights

                kwargs[
                    "views"
                ] = np.zeros(1)

                kwargs[
                    "view_matrix"
                ] = np.ones(
                    (
                        1,
                        len(expected_returns)
                    )
                )

            weights = optimizer(
                **kwargs
            )

            optimizer_outputs[
                method_name
            ] = weights

            assert isinstance(
                weights,
                pd.Series
            )

            assert len(weights) == 5

            assert np.isfinite(
                weights.to_numpy(
                    dtype=float
                )
            ).all()

            assert np.isclose(
                weights.sum(),
                1.0,
                atol=1e-5
            )

            assert (
                weights.to_numpy()
                >= -1e-8
            ).all()

        run_test(
            method_name,
            optimizer_test
        )


# ============================================================
# 10. RANDOMIZED OPTIMIZATION STRESS
# ============================================================

section("6. RANDOMIZED OPTIMIZATION STRESS")

if optimization_module is not None:

    for n_assets in [
        2,
        3,
        5,
        10,
        20
    ]:

        for seed in [
            1,
            2,
            3,
            4,
            5
        ]:

            def randomized_test(
                n_assets=n_assets,
                seed=seed
            ):

                prices = make_price_data(
                    n_assets=n_assets,
                    n_days=600,
                    seed=seed + 1000
                )

                returns = (
                    prices
                    .pct_change()
                    .dropna()
                )

                covariance = (
                    returns.cov()
                    * 252
                )

                expected_returns = (
                    returns.mean()
                    * 252
                )

                market_weights = pd.Series(
                    np.ones(n_assets)
                    / n_assets,
                    index=returns.columns
                )

                for (
                    method_name,
                    optimizer
                ) in optimizer_functions.items():

                    kwargs = {
                        "expected_returns":
                            expected_returns,

                        "covariance":
                            covariance
                    }

                    if (
                        method_name
                        == "Black-Litterman"
                    ):

                        kwargs[
                            "market_weights"
                        ] = market_weights

                        kwargs[
                            "views"
                        ] = np.zeros(1)

                        kwargs[
                            "view_matrix"
                        ] = np.ones(
                            (
                                1,
                                n_assets
                            )
                        )

                    weights = optimizer(
                        **kwargs
                    )

                    assert np.isfinite(
                        weights.to_numpy(
                            dtype=float
                        )
                    ).all()

                    assert np.isclose(
                        weights.sum(),
                        1.0,
                        atol=1e-5
                    )

                    assert (
                        weights.to_numpy()
                        >= -1e-8
                    ).all()

            run_test(
                f"{n_assets} assets / seed {seed}",
                randomized_test
            )


# ============================================================
# 11. OPTIMIZATION EDGE CASES
# ============================================================

section("7. OPTIMIZATION EDGE CASES")

if optimization_module is not None:

    def test_two_assets():

        prices = make_price_data(
            n_assets=2,
            n_days=500,
            seed=5000
        )

        returns = (
            prices
            .pct_change()
            .dropna()
        )

        covariance = (
            returns.cov()
            * 252
        )

        expected_returns = (
            returns.mean()
            * 252
        )

        weights = (
            optimization_module
            .minimum_variance(
                expected_returns=expected_returns,
                covariance=covariance
            )
        )

        assert np.isclose(
            weights.sum(),
            1.0,
            atol=1e-5
        )

    run_test(
        "Two-asset minimum variance",
        test_two_assets
    )


    def test_high_correlation():

        rng = np.random.default_rng(
            6000
        )

        base = rng.normal(
            0,
            0.01,
            1000
        )

        returns = pd.DataFrame(
            {
                "A": base,

                "B":
                    base
                    + rng.normal(
                        0,
                        0.0001,
                        1000
                    ),

                "C":
                    rng.normal(
                        0,
                        0.01,
                        1000
                    )
            }
        )

        covariance = (
            returns.cov()
            * 252
        )

        expected_returns = (
            returns.mean()
            * 252
        )

        for optimizer in [

            optimization_module
            .minimum_variance,

            optimization_module
            .maximum_sharpe,

            optimization_module
            .risk_parity,

            optimization_module
            .hierarchical_risk_parity
        ]:

            weights = optimizer(
                expected_returns=expected_returns,
                covariance=covariance
            )

            assert np.isfinite(
                weights.to_numpy(
                    dtype=float
                )
            ).all()

            assert np.isclose(
                weights.sum(),
                1.0,
                atol=1e-5
            )

    run_test(
        "Highly correlated assets",
        test_high_correlation
    )


# ============================================================
# 12. BACKTEST ENGINE
# ============================================================

section("8. BACKTEST ENGINE")

if (
    backtest_module is not None
    and optimization_module is not None
):

    prices = make_price_data(
        n_assets=5,
        n_days=1000,
        seed=7000
    )

    selected_assets = list(
        prices.columns
    )

    backtest_results = {}


    for method_name, optimizer in (
        optimizer_functions.items()
    ):

        def backtest_test(
            method_name=method_name,
            optimizer=optimizer
        ):

            result = (
                backtest_module
                .run_backtest(
                    prices=prices,
                    selected_assets=selected_assets,
                    optimizer=optimizer,
                    benchmark_prices=None,
                    train_window=252,
                    rebalance_frequency="M",
                    max_turnover=0.25,
                    initial_capital=1.0
                )
            )

            backtest_results[
                method_name
            ] = result

            assert isinstance(
                result,
                dict
            )

            assert (
                "portfolio_returns"
                in result
            )

            portfolio_returns = (
                result[
                    "portfolio_returns"
                ]
            )

            assert len(
                portfolio_returns
            ) > 0

            assert np.isfinite(
                portfolio_returns.to_numpy(
                    dtype=float
                )
            ).all()

        run_test(
            method_name,
            backtest_test
        )


# ============================================================
# 13. BACKTEST MATHEMATICAL TESTS
# ============================================================

section("9. BACKTEST MATHEMATICAL TESTS")

if backtest_module is not None:

    def test_nav_identity():

        returns = pd.Series(
            [
                0.10,
                -0.05,
                0.20
            ]
        )

        nav = (
            backtest_module
            .calculate_nav(
                returns,
                initial_value=1.0
            )
        )

        expected = (
            1.10
            * 0.95
            * 1.20
        )

        assert np.isclose(
            nav.iloc[-1],
            expected
        )

    run_test(
        "NAV compounding identity",
        test_nav_identity
    )


    def test_turnover_identity():

        old_weights = pd.Series(
            {
                "A": 0.50,
                "B": 0.50
            }
        )

        new_weights = pd.Series(
            {
                "A": 0.70,
                "B": 0.30
            }
        )

        turnover = (
            backtest_module
            .calculate_turnover(
                old_weights,
                new_weights
            )
        )

        assert np.isclose(
            turnover,
            0.20
        )

    run_test(
        "One-way turnover identity",
        test_turnover_identity
    )


    def test_portfolio_return_identity():

        asset_returns = pd.DataFrame(
            {
                "A": [
                    0.10,
                    0.02
                ],
                "B": [
                    0.00,
                    0.04
                ]
            }
        )

        weights = pd.Series(
            {
                "A": 0.60,
                "B": 0.40
            }
        )

        result = (
            backtest_module
            .calculate_portfolio_returns(
                asset_returns,
                weights
            )
        )

        expected = pd.Series(
            [
                0.06,
                0.028
            ]
        )

        assert np.allclose(
            result.to_numpy(),
            expected.to_numpy()
        )

    run_test(
        "Portfolio return identity",
        test_portfolio_return_identity
    )


# ============================================================
# 14. PERFORMANCE ENGINE
# ============================================================

section("10. PERFORMANCE ENGINE")

if performance_module is not None:

    rng = np.random.default_rng(
        8000
    )

    dates = pd.bdate_range(
        start="2020-01-01",
        periods=1000
    )

    portfolio_returns = {}

    for name in optimizer_functions:

        portfolio_returns[
            name
        ] = pd.Series(
            rng.normal(
                0.0004,
                0.01,
                len(dates)
            ),
            index=dates
        )

    benchmark_returns = pd.Series(
        rng.normal(
            0.00045,
            0.011,
            len(dates)
        ),
        index=dates
    )


    def test_total_return():

        returns = pd.Series(
            [
                0.10,
                -0.05,
                0.10
            ]
        )

        result = (
            performance_module
            .calculate_total_return(
                returns
            )
        )

        expected = (
            1.10
            * 0.95
            * 1.10
        ) - 1

        assert np.isclose(
            result,
            expected
        )

    run_test(
        "Total return identity",
        test_total_return
    )


    def test_performance_nav():

        returns = pd.Series(
            [
                0.10,
                -0.05,
                0.10
            ]
        )

        nav = (
            performance_module
            .calculate_nav(
                returns
            )
        )

        expected = (
            1.10
            * 0.95
            * 1.10
        )

        assert np.isclose(
            nav.iloc[-1],
            expected
        )

    run_test(
        "Performance NAV identity",
        test_performance_nav
    )


    def test_drawdown():

        nav = pd.Series(
            [
                1.0,
                1.2,
                1.0,
                1.3
            ]
        )

        drawdown = (
            performance_module
            .calculate_drawdown(
                nav
            )
        )

        expected = (
            1.0 / 1.2
        ) - 1

        assert np.isclose(
            drawdown.min(),
            expected
        )

    run_test(
        "Maximum drawdown identity",
        test_drawdown
    )


    def test_performance_comparison():

        result = (
            performance_module
            .compare_portfolios(
                portfolio_returns,
                benchmark_returns
            )
        )

        assert result.shape[0] == 6

        required_columns = [
            "Total Return",
            "CAGR",
            "Annualized Volatility",
            "Sharpe Ratio",
            "Sortino Ratio",
            "Maximum Drawdown",
            "Calmar Ratio"
        ]

        for column in required_columns:

            assert column in result.columns

        values = result[
            required_columns
        ].values

        assert np.isfinite(
            values
        ).all()

    run_test(
        "Six portfolio performance comparison",
        test_performance_comparison
    )


    def test_performance_ranking():

        comparison = (
            performance_module
            .compare_portfolios(
                portfolio_returns,
                benchmark_returns
            )
        )

        ranking = (
            performance_module
            .rank_portfolios(
                comparison
            )
        )

        assert len(
            ranking
        ) == 6

        assert (
            "Overall Rank"
            in ranking.columns
        )

        assert ranking[
            "Overall Rank"
        ].notna().all()

    run_test(
        "Portfolio ranking",
        test_performance_ranking
    )


# ============================================================
# 15. PERFORMANCE EDGE CASES
# ============================================================

section("11. PERFORMANCE EDGE CASES")

if performance_module is not None:

    def test_constant_returns():

        returns = pd.Series(
            np.zeros(100)
        )

        volatility = (
            performance_module
            .calculate_annualized_volatility(
                returns
            )
        )

        assert np.isclose(
            volatility,
            0.0
        )

    run_test(
        "Constant returns",
        test_constant_returns
    )


    def test_nan_rejection():

        returns = pd.Series(
            [
                0.01,
                np.nan,
                0.02
            ]
        )

        try:

            performance_module.calculate_total_return(
                returns
            )

        except ValueError:

            return

        raise AssertionError(
            "NaN should be rejected."
        )

    run_test(
        "NaN rejection",
        test_nan_rejection
    )


    def test_inf_rejection():

        returns = pd.Series(
            [
                0.01,
                np.inf,
                0.02
            ]
        )

        try:

            performance_module.calculate_total_return(
                returns
            )

        except ValueError:

            return

        raise AssertionError(
            "Infinity should be rejected."
        )

    run_test(
        "Infinite value rejection",
        test_inf_rejection
    )


# ============================================================
# 16. PORTFOLIO RISK EDGE CASES
# ============================================================

section("12. PORTFOLIO RISK EDGE CASES")

if portfolio_risk_module is not None:

    def test_negative_weights():

        weights = pd.Series(
            {
                "A": 1.2,
                "B": -0.2
            }
        )

        covariance = pd.DataFrame(
            [
                [0.04, 0.01],
                [0.01, 0.03]
            ],
            index=[
                "A",
                "B"
            ],
            columns=[
                "A",
                "B"
            ]
        )

        try:

            portfolio_risk_module.portfolio_variance(
                weights,
                covariance
            )

        except ValueError:

            return

        raise AssertionError(
            "Negative weights should be rejected."
        )

    run_test(
        "Negative weight rejection",
        test_negative_weights
    )


    def test_zero_weights():

        weights = pd.Series(
            {
                "A": 0.0,
                "B": 0.0
            }
        )

        covariance = pd.DataFrame(
            [
                [0.04, 0.01],
                [0.01, 0.03]
            ],
            index=[
                "A",
                "B"
            ],
            columns=[
                "A",
                "B"
            ]
        )

        try:

            portfolio_risk_module.portfolio_variance(
                weights,
                covariance
            )

        except ValueError:

            return

        raise AssertionError(
            "Zero-sum weights should be rejected."
        )

    run_test(
        "Zero-sum weight rejection",
        test_zero_weights
    )


# ============================================================
# 17. LOOK-AHEAD / DATA LEAKAGE
# ============================================================

section("13. LOOK-AHEAD / DATA LEAKAGE SANITY TEST")

if backtest_module is not None:

    def test_future_data_does_not_change_past():

        prices = make_price_data(
            n_assets=5,
            n_days=800,
            seed=9000
        )

        cutoff = 600

        original = prices.copy()

        modified = prices.copy()

        modified.iloc[
            cutoff:
        ] *= 10.0

        original_history = (
            backtest_module
            .calculate_returns(
                original.iloc[
                    :cutoff
                ]
            )
        )

        modified_history = (
            backtest_module
            .calculate_returns(
                modified.iloc[
                    :cutoff
                ]
            )
        )

        assert np.allclose(
            original_history.to_numpy(),
            modified_history.to_numpy(),
            equal_nan=True
        )

    run_test(
        "Future prices do not change historical returns",
        test_future_data_does_not_change_past
    )


# ============================================================
# 18. LARGE DATASET STRESS
# ============================================================

section("14. LARGE DATASET STRESS")

if optimization_module is not None:

    def large_dataset_test():

        prices = make_price_data(
            n_assets=20,
            n_days=3000,
            seed=10000
        )

        returns = (
            prices
            .pct_change()
            .dropna()
        )

        covariance = (
            returns.cov()
            * 252
        )

        expected_returns = (
            returns.mean()
            * 252
        )

        market_weights = pd.Series(
            np.ones(20) / 20,
            index=returns.columns
        )

        for (
            method_name,
            optimizer
        ) in optimizer_functions.items():

            kwargs = {
                "expected_returns":
                    expected_returns,

                "covariance":
                    covariance
            }

            if (
                method_name
                == "Black-Litterman"
            ):

                kwargs[
                    "market_weights"
                ] = market_weights

                kwargs[
                    "views"
                ] = np.zeros(1)

                kwargs[
                    "view_matrix"
                ] = np.ones(
                    (
                        1,
                        20
                    )
                )

            weights = optimizer(
                **kwargs
            )

            assert np.isfinite(
                weights.to_numpy(
                    dtype=float
                )
            ).all()

            assert np.isclose(
                weights.sum(),
                1.0,
                atol=1e-5
            )

            assert (
                weights.to_numpy()
                >= -1e-8
            ).all()

    run_test(
        "20 assets / 3000 observations",
        large_dataset_test
    )


# ============================================================
# 19. END-TO-END INTEGRATION
# ============================================================

section("15. END-TO-END INTEGRATION TEST")

if (
    backtest_module is not None
    and performance_module is not None
    and optimization_module is not None
):

    def integration_test():

        prices = make_price_data(
            n_assets=5,
            n_days=1000,
            seed=11000
        )

        assets = list(
            prices.columns
        )

        results = {}

        for (
            method_name,
            optimizer
        ) in optimizer_functions.items():

            result = (
                backtest_module
                .run_backtest(
                    prices=prices,
                    selected_assets=assets,
                    optimizer=optimizer,
                    benchmark_prices=None,
                    train_window=252,
                    rebalance_frequency="M",
                    max_turnover=0.25,
                    initial_capital=1.0
                )
            )

            results[
                method_name
            ] = result

        portfolio_returns = {
            name:
                result[
                    "portfolio_returns"
                ]
            for name, result
            in results.items()
        }

        performance = (
            performance_module
            .compare_portfolios(
                portfolio_returns
            )
        )

        ranking = (
            performance_module
            .rank_portfolios(
                performance
            )
        )

        assert len(results) == 6

        assert len(
            portfolio_returns
        ) == 6

        assert len(
            performance
        ) == 6

        assert len(
            ranking
        ) == 6

        assert (
            performance[
                "CAGR"
            ].notna().all()
        )

        assert (
            performance[
                "Sharpe Ratio"
            ].notna().all()
        )

        assert (
            performance[
                "Maximum Drawdown"
            ].notna().all()
        )

    run_test(
        "Optimization → Backtest → Performance",
        integration_test
    )


# ============================================================
# 20. REAL DATA TEST
# ============================================================

section("16. REAL-DATA TEST")

print(
    """
This section uses Yahoo Finance if
yfinance is installed.
"""
)

try:

    import yfinance as yf

    real_assets = [
        "RELIANCE.NS",
        "TCS.NS",
        "INFY.NS",
        "HDFCBANK.NS",
        "ICICIBANK.NS"
    ]

    real_benchmark = "^CRSLDX"


    def real_data_test():

        prices = yf.download(
            real_assets,
            start="2022-01-01",
            end="2026-01-01",
            auto_adjust=True,
            progress=False
        )

        if isinstance(
            prices.columns,
            pd.MultiIndex
        ):

            prices = prices["Close"]

        prices = prices.dropna(
            how="all"
        )

        assert not prices.empty

        available_assets = [
            asset
            for asset in real_assets
            if asset in prices.columns
        ]

        assert len(
            available_assets
        ) == 5

        prices = prices[
            available_assets
        ]

        assert not prices.isna().all().any()

        benchmark_prices = yf.download(
            real_benchmark,
            start="2022-01-01",
            end="2026-01-01",
            auto_adjust=True,
            progress=False
        )

        if isinstance(
            benchmark_prices.columns,
            pd.MultiIndex
        ):

            benchmark_prices = (
                benchmark_prices["Close"]
            )

        benchmark_prices = (
            benchmark_prices
            .dropna()
        )

        assert not benchmark_prices.empty

        print(
            "\n    Real price observations:",
            len(prices)
        )

        print(
            "    Real benchmark observations:",
            len(benchmark_prices)
        )

    run_test(
        "Yahoo Finance real-data availability",
        real_data_test
    )

except ImportError:

    print(
        "    🟡 yfinance not installed - "
        "real-data test skipped."
    )


# ============================================================
# 21. FINAL REPORT
# ============================================================

section("FINAL STRESS TEST REPORT")

print(
    f"Tests run:       {TESTS_RUN}"
)

print(
    f"Tests passed:    {TESTS_PASSED}"
)

print(
    f"Tests failed:    {TESTS_FAILED}"
)

if TESTS_RUN > 0:

    pass_rate = (
        TESTS_PASSED
        / TESTS_RUN
        * 100
    )

    print(
        f"Pass rate:       {pass_rate:.2f}%"
    )


# ============================================================
# FAILURE REPORT
# ============================================================

if FAILURES:

    print()
    print(
        "=" * 75
    )

    print(
        "🔴 FAILED TESTS"
    )

    print(
        "=" * 75
    )

    for i, failure in enumerate(
        FAILURES,
        start=1
    ):

        print()
        print(
            f"{i}. {failure['test']}"
        )

        print(
            f"   {failure['error']}"
        )


# ============================================================
# FINAL STATUS
# ============================================================

print()
print(
    "=" * 75
)

if TESTS_FAILED == 0:

    print(
        "🔥🔥🔥 ALL STRESS TESTS PASSED 🔥🔥🔥"
    )

    print()
    print(
        "The project passed the automated"
    )

    print(
        "computational, edge-case, integration,"
    )

    print(
        "and data-leakage tests."
    )

    print()
    print(
        "NEXT STEP: DASHBOARD"
    )

else:

    print(
        "🔴 STRESS TEST FAILED"
    )

    print()
    print(
        "Fix every failed test first."
    )

print(
    "=" * 75
)

POR-DASHBOARD FULL STRESS TEST

Current working directory:
C:\Users\Ak\PycharmProjects\POR-Dashboard\notebooks
Project root:
C:\Users\Ak\PycharmProjects\POR-Dashboard

1. MODULE IMPORT TEST
    🟢 PASS: src.returns.returns
    🟢 PASS: src.risk.risk
    🟢 PASS: src.risk.portfolio_risk
    🟢 PASS: src.optimization.optimization
    🟢 PASS: src.backtest.backtest
    🟢 PASS: src.performance.performance

2. RETURNS ENGINE
    🟢 PASS: Simple returns
    🟢 PASS: Log returns
    🟢 PASS: Return matrix
    🟢 PASS: Log return matrix
    🟢 PASS: Cumulative return identity
    🟢 PASS: Historical expected return
    🟢 PASS: Geometric expected return

3. RISK ENGINE
    🟢 PASS: Historical volatility
    🟢 PASS: EWMA volatility
    🟢 PASS: Sample covariance
    🟢 PASS: Ledoit-Wolf covariance
    🟢 PASS: Correlation matrix

4. PORTFOLIO RISK ENGINE
    🟢 PASS: Portfolio variance identity
    🟢 PASS: Portfolio volatility identity
    🟢 PASS: Marginal risk
    🟢 PASS: Component risk sums to portfolio volat